# RSO-114: Analyze airflow behavior and overall thermal trends

During the shutdown period (31.1.26 -14.2.26) we ran multiple versions (different louver configurations) of BLOCK-T679. This notebook analyzes the thermal behavior for different airflow conditions.

**Description**

Use airflow measurements (when available) to see whether louver settings actually change ventilation, and summarize what seems to work better or worse.

**Expected results:**

Plots: airflow vs time and airflow vs temperature change

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from astropy.time import Time, TimeDelta
from datetime import timezone

from lsst.summit.utils.efdUtils import getEfdData, makeEfdClient
from lsst.summit.utils.tmaUtils import TMAEventMaker, TMAState

In [ ]:
t_start_period = Time("2026-01-31T12:00:00Z", scale="utc")
t_end_period = Time("2026-02-15T12:00:00Z", scale="utc")

efd_client = makeEfdClient()

### Queries

In [ ]:
def query_setlouvers(start, end):
    df_louvers = getEfdData(
        client=efd_client,
        topic="lsst.sal.MTDome.command_setLouvers",
        columns=["*"],
        begin=start,
        end=end,
    )

    return df_louvers

### Configuration of louvers

In [ ]:
df_setlouvers = query_setlouvers(t_start_period, t_end_period)

In [ ]:
# Copy current index into a new column before any merge
df_setlouvers['time_stamp'] = df_setlouvers.index

In [ ]:
# Select all columns that start with "position"
position_cols = df_setlouvers.filter(regex=r'^position').columns

# Sort columns
position_cols = sorted(position_cols, key=lambda x: int(x.replace('position', '')))

# Compute unique combinations
combination_counts = (
    df_setlouvers[position_cols]
    .value_counts()
    .reset_index(name='count')
)

# Create configuration ID (1 to N)
combination_counts['louvers_conf'] = range(1, len(combination_counts) + 1)

# Merge configuration ID back into original dataframe
df_setlouvers = df_setlouvers.merge(
    combination_counts[position_cols + ['louvers_conf']],
    on=position_cols,
    how='left'
)

print(f"Number of unique configurations detected: {len(combination_counts)}\n")

# Print configurations showing only non-zero positions
for _, row in combination_counts.iterrows():
    
    conf_id = row['louvers_conf']
    count = row['count']
    
    print(f"Configuration {conf_id} (appears {count} times):")
    
    # Extract position values
    config = row[position_cols]
    
    # Keep only non-zero values
    non_zero = config[config != 0]
    
    if len(non_zero) == 0:
        print("  All positions are 0")
    else:
        for col, val in non_zero.items():
        #for col, val in config.items():
            print(f"  {col}: {val}")
    
    print("-" * 40)

15 combinations have been made, varying the opening of the louvers: 2, 11, 12, 20, 21, and 29.

In [ ]:
df_setlouvers['duration'] = (
    df_setlouvers['time_stamp'].shift(-1) - df_setlouvers['time_stamp']
)
last_idx = df_setlouvers.index[-1]
df_setlouvers.loc[last_idx, 'duration'] = (
    pd.Timestamp(t_end_period.to_datetime(), tz="UTC")
    - df_setlouvers.loc[last_idx, 'time_stamp']
)
df_setlouvers['duration_minutes'] = df_setlouvers['duration'].dt.total_seconds() / 60

### Select cases within configurations and make plots

In [ ]:
louvers_used = '[2,11,12,20,21,29]'
config_description = {1:f'{louvers_used} 100%',
                      2:f'{louvers_used} 10%', 
                      3:f'{louvers_used} 50%',
                      4:f'[11,21,29] 100%',
                      5:f'[2,11,12] 100%',
                      6:f'[20,21,29] 100%',
                      7:f'[11,12,20,21,29] 50%',
                      8:f'[2,12,20,21,29] 50% -- 11 100%',
                      9:f'[2,12,20,21,29] 50%',
                      10:f'[11,20,29] 100%',
                      11:f'{louvers_used} 30%',
                      12:f'[11,12,20,21,29] 50% -- 2 100%',
                      13:f'{louvers_used} 60%',
                      14:f'{louvers_used} 0%',
                      15:f'[11,20,29] 50%',
                     }

for cfg in range(1,16): #corresponding to the 15 different configurations found
    df_air_tma_list = []
    df_air_pXpY_list = []
    df_air_mXmY_list = []
    df_air_pXmY_list = []
    df_air_mXpY_list = []
    df_air_weat_list = []
    df_temp_m1m3_list = []
    df_temp_cam_list = []
    df_temp_m2_list = []
    df_temp_weat_list = []
    
    condition = df_setlouvers['duration_minutes'].between(20, 100, inclusive="neither") & (df_setlouvers['louvers_conf'] == cfg)
    df_selected = df_setlouvers[condition].copy()
    df_selected['end_time_stamp'] = df_selected['time_stamp'] + df_selected['duration']
    
    #retrieve data for this configuration
    print(f"Number of tests selected for configuration {cfg}:",len(df_selected))
    # ['AT Dome' 'MTDome-ESS10' 'MTDome-ESS12' 'Weather tower']
    for row in df_selected.itertuples():
        begin = Time(row.time_stamp)
        end = Time(row.end_time_stamp)
        df_airf = getEfdData(
            client=efd_client,
            topic="lsst.sal.ESS.airFlow", #seems that it comes from the weather tower
            columns=["location","direction","speed"],
            begin=begin,
            end=end,
        )
        df_airt = getEfdData(
            client=efd_client,
            topic="lsst.sal.ESS.airTurbulence", #seems that these carry the indoor anemometers
            columns=["sensorName","location","speedMagnitude","sonicTemperatureStdDev"],
            begin=begin,
            end=end,
        )
        df_temp = getEfdData(
            client=efd_client,
            topic="lsst.sal.ESS.temperature", 
            columns=["sensorName","temperatureItem0"],
            begin=begin,
            end=end,
        )
        dfa_tma = df_airt[df_airt["location"]=='TMA (unknown location)'].copy()
        dfa_pXpY = df_airt[df_airt["location"]=='TEA (+X/+Y)'].copy()
        dfa_mXmY = df_airt[df_airt["location"]=='TEA (-X/-Y)'].copy()
        dfa_pXmY = df_airt[df_airt["location"]=='TEA (+X/-Y)'].copy()
        dfa_mXpY = df_airt[df_airt["location"]=='TEA (-X/+Y)'].copy()
        dfa_weat = df_airf[df_airf["location"]=='Weather tower'].copy()
        dft_m1m3 = df_temp[df_temp["sensorName"]=='M1M3-ESS03'].copy()
        dft_cam = df_temp[df_temp["sensorName"]=='Camera-ESS01'].copy()
        dft_m2 = df_temp[df_temp["sensorName"]=='M2-ESS02'].copy()
        dft_weat = df_temp[df_temp["sensorName"]=='Weather tower air temperature'].copy()
        df_air_tma_list.append(dfa_tma)
        df_air_pXpY_list.append(dfa_pXpY)
        df_air_mXmY_list.append(dfa_mXmY)
        df_air_pXmY_list.append(dfa_pXmY)
        df_air_mXpY_list.append(dfa_mXpY)
        df_air_weat_list.append(dfa_weat)
        df_temp_m1m3_list.append(dft_m1m3)
        df_temp_cam_list.append(dft_cam)
        df_temp_m2_list.append(dft_m2)
        df_temp_weat_list.append(dft_weat)
    
    #and now make plots
    for i,(dfa_tma,dfa_pXpY,dfa_mXmY,dfa_pXmY,dfa_mXpY,dfa_weat,dft_m1m3,dft_m2,dft_cam,dft_weat) in enumerate(zip(df_air_tma_list,df_air_pXpY_list,df_air_mXmY_list,df_air_pXmY_list,df_air_mXpY_list,df_air_weat_list,df_temp_m1m3_list,df_temp_m2_list,df_temp_cam_list,df_temp_weat_list)):
        fig, ax1 = plt.subplots()
        t0 = dfa_tma.index[0]
        t_tma_air = (dfa_tma.index - t0).total_seconds()
        t_pXpY_air = (dfa_pXpY.index - t0).total_seconds()
        t_mXmY_air = (dfa_mXmY.index - t0).total_seconds()
        t_pXmY_air = (dfa_pXmY.index - t0).total_seconds()
        t_mXpY_air = (dfa_mXpY.index - t0).total_seconds()
        t_weat_air = (dfa_weat.index - t0).total_seconds()
        t_m1m3_temp = (dft_m1m3.index - t0).total_seconds()
        t_m2_temp = (dft_m2.index - t0).total_seconds()
        t_cam_temp = (dft_cam.index - t0).total_seconds()
        t_weat_temp = (dft_weat.index - t0).total_seconds()
        ax1.plot(t_weat_air[::5], dfa_weat["speed"].iloc[::5],marker='.',label='Weather tower',color='darkviolet')
        ax1.plot(t_tma_air[::5], dfa_tma["speedMagnitude"].iloc[::5],marker='.',label='TMA',color='blue')
        ax1.set_xlabel("Time since start (s)")
        ax1.set_ylabel("Airflow speed (m/s)")
        ax1.plot(t_pXpY_air[::5], dfa_pXpY["speedMagnitude"].iloc[::5],marker='+',label='ESS 125 +X/+Y',color='red')
        ax1.plot(t_mXmY_air[::5], dfa_mXmY["speedMagnitude"].iloc[::5],marker='_',label='ESS 123 -X/-Y',color='orange')
        ax1.plot(t_pXmY_air[::5], dfa_pXmY["speedMagnitude"].iloc[::5],marker='x',label='ESS 124 +X/-Y',color='tan')
        ax1.plot(t_mXpY_air[::5], dfa_mXpY["speedMagnitude"].iloc[::5],marker='X',label='ESS 126 -X/+Y',color='goldenrod')
        fig.suptitle(f"Config. {cfg} -- {config_description[cfg]}")
        fig.legend()
        plt.savefig(f"plots/config_{cfg}_case_{i}.png")
    plt.close()


### Make plots per day

In [ ]:
one_day = TimeDelta(1, format='jd')  # 1 day
sampling = 300

day_starts = []
t = t_start_period
while t < t_end_period:
    day_starts.append(t)
    t += one_day

for day_idx, day_start in enumerate(day_starts):
    day_end = day_start + one_day    
    day_start_dt = day_start.to_datetime().replace(tzinfo=timezone.utc)
    day_end_dt   = day_end.to_datetime().replace(tzinfo=timezone.utc)

    print(f"Processing day {day_idx}: {day_start.isot} - {day_end.isot}")

    condition = ((df_setlouvers['time_stamp'] >= day_start_dt) &
    (df_setlouvers['time_stamp'] < day_end_dt))

    df_selected = df_setlouvers[condition].copy()
    df_selected['end_time_stamp'] = df_selected['time_stamp'] + df_selected['duration']

    df_air_tma_list = []
    df_air_pXpY_list = []
    df_air_mXmY_list = []
    df_air_pXmY_list = []
    df_air_mXpY_list = []
    df_air_weat_list = []

    fig, ax1 = plt.subplots()

    for i,row in enumerate(df_selected.itertuples()): 
        begin = Time(row.time_stamp)
        end = Time(row.end_time_stamp)
        print(begin, end)

        df_airf = getEfdData(
            client=efd_client,
            topic="lsst.sal.ESS.airFlow", #seems that it comes from the weather tower
            columns=["location","direction","speed"],
            begin=begin,
            end=end,
        )
        df_airt = getEfdData(
            client=efd_client,
            topic="lsst.sal.ESS.airTurbulence", #seems that these carry the indoor anemometers
            columns=["sensorName","location","speedMagnitude","sonicTemperatureStdDev"],
            begin=begin,
            end=end,
        )

        dfa_tma = df_airt[df_airt["location"]=='TMA (unknown location)'].copy()
        dfa_pXpY = df_airt[df_airt["location"]=='TEA (+X/+Y)'].copy()
        dfa_mXmY = df_airt[df_airt["location"]=='TEA (-X/-Y)'].copy()
        dfa_pXmY = df_airt[df_airt["location"]=='TEA (+X/-Y)'].copy()
        dfa_mXpY = df_airt[df_airt["location"]=='TEA (-X/+Y)'].copy()
        dfa_weat = df_airf[df_airf["location"]=='Weather tower'].copy()

        df_air_tma_list.append(dfa_tma)
        df_air_weat_list.append(dfa_weat)
        df_air_pXpY_list.append(dfa_pXpY)
        df_air_mXmY_list.append(dfa_mXmY)
        df_air_pXmY_list.append(dfa_pXmY)
        df_air_mXpY_list.append(dfa_mXpY)

        dfa_tma_all = pd.concat(df_air_tma_list).sort_index()
        dfa_weat_all = pd.concat(df_air_weat_list).sort_index()
        dfa_pXpY_all = pd.concat(df_air_pXpY_list).sort_index()
        dfa_mXmY_all = pd.concat(df_air_mXmY_list).sort_index()
        dfa_pXmY_all = pd.concat(df_air_pXmY_list).sort_index()
        dfa_mXpY_all = pd.concat(df_air_mXpY_list).sort_index()

        start = pd.Timestamp(row.time_stamp)
        ax1.axvline(start, color='gray', linestyle='--', alpha=0.5)
        ax1.text(start, 1.02,
            f"{row.louvers_conf}",
            rotation=0, fontsize=8, ha='center', va='bottom',
            transform=ax1.get_xaxis_transform(),
            clip_on=False)
            #verticalalignment='bottom', alpha=0.7)

    ax1.plot(dfa_weat_all.index[::sampling], dfa_weat_all["speed"].iloc[::sampling],
        marker='.', label='Weather tower', color='darkviolet')
        
    ax1.plot(dfa_tma_all.index[::sampling], dfa_tma_all["speedMagnitude"].iloc[::sampling],
        marker='.', label='TMA', color='blue')
        
    ax1.plot(dfa_pXpY_all.index[::sampling], dfa_pXpY_all["speedMagnitude"].iloc[::sampling],
        marker='+',label='ESS 125 +X/+Y',color='red')
        
    ax1.plot(dfa_mXmY_all.index[::sampling], dfa_mXmY_all["speedMagnitude"].iloc[::sampling],
        marker='_',label='ESS 123 -X/-Y',color='orange')
        
    ax1.plot(dfa_pXmY_all.index[::sampling], dfa_pXmY_all["speedMagnitude"].iloc[::sampling],
        marker='x',label='ESS 124 +X/-Y',color='tan')
        
    ax1.plot(dfa_mXpY_all.index[::sampling], dfa_mXpY_all["speedMagnitude"].iloc[::sampling],
        marker='X',label='ESS 126 -X/+Y',color='goldenrod')

        
    ax1.set_xlabel("Time (UTC)")
    ax1.set_ylabel("Airflow speed (m/s)")
    plt.setp(ax1.get_xticklabels(), rotation=30, ha='right')
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax1.xaxis.set_major_locator(mdates.HourLocator(interval=2))
        
    fig.suptitle(f"{day_start.isot[:10]}")
    fig.legend()
    plt.tight_layout()
    plt.savefig(f"plots/day_{day_idx}.png", bbox_inches='tight')
    plt.close()
